# Пример разбиения на группы (AASplitter)

В этом ноутбуке показано, как работает `AASplitter` — базовый сплиттер для разбиения данных на control/test группы.

In [1]:
from hypex import AATest
from hypex.dataset import (
    Dataset,
    SmallDataset,
    InfoRole,
    TargetRole,
    TreatmentRole,
    DatasetAdapter,
    DefaultRole,
    StratificationRole
)
from hypex.dataset.roles import TargetRole, FeatureRole, InfoRole, StatisticRole
from hypex.comparators import StatsTTest, StatsChi2Test
from hypex.utils import create_test_data
from hypex.splitters import AASplitter
from hypex.comparators import GroupSizes
import pyspark.sql as spark
import pandas as pd

ModuleNotFoundError: No module named 'hypex'

## 1. Создание тестового датасета

Создадим синтетические данные с пользователями и их метриками.

In [75]:
session = (
            spark
            .SparkSession.builder
            .master("local[*]")
            .config("spark.driver.memory", "4g")
            .config("spark.executo|r.memory", "4g")
            .config("spark.memory.fraction", "0.8") 
            .config("spark.memory.storageFraction", "0.3") 
            .getOrCreate()
          )
data = (
            session.read.format('csv')
            .option("header", "true")
            .option("inferSchema", "true")
            .option("ignoreLeadingWhiteSpace", "true") 
            .option("ignoreTrailingWhiteSpace", "true")
            .load("../data_no_index.csv")
        )
# data = pd.read_csv("../data_no_index.csv")
ds = Dataset(
    roles={
        "user_id": InfoRole(float),
        "treat": TreatmentRole(),
        "pre_spends": TargetRole(),
        "post_spends": DefaultRole(),
        "gender": StratificationRole(str),
        "industry" : DefaultRole()
    }, 
    data=data, 
    session=session
    )

ds

/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all d

,user_id,signup_month,treat,pre_spends,post_spends,age,gender,industry
0,0.0,0.0,0.0,519.5,424.666667,39.0,M,Logistics
1,1.0,0.0,0.0,500.5,423.111111,19.0,F,Logistics
2,2.0,0.0,0.0,499.0,423.444444,29.0,M,E-commerce
3,3.0,2.0,1.0,494.0,530.555556,38.0,M,E-commerce
4,4.0,9.0,1.0,455.0,451.888889,53.0,F,Logistics
...,...,...,...,...,...,...,...,...
9995,9995.0,0.0,0.0,490.0,429.555556,26.0,F,Logistics
9996,9996.0,0.0,0.0,464.5,414.666667,49.0,M,E-commerce
9997,9997.0,0.0,0.0,498.0,411.666667,30.0,F,E-commerce
9998,9998.0,2.0,1.0,528.0,516.555556,36.0,F,Logistics


## 2. Запуск AASplitter

`AASplitter` создаёт колонку с метками групп (control, test_0, test_1, ...).

Параметры:
- `control_size` — доля контрольной группы (по умолчанию 0.5)
- `random_state` — seed для воспроизводимости
- `sample_size` — доля данных для эксперимента (по умолчанию 1.0)

In [76]:
from hypex.dataset import ExperimentData, AdditionalTreatmentRole

# Оборачиваем в ExperimentData для работы пайплайна
exp_data = ExperimentData(ds)

# Создаём и запускаем сплиттер
splitter = AASplitter(
    control_size=0.5,
    random_state=21,
    constant_key=True,
    save_groups=True  # Сохраняем группы в data.groups для удобного доступа
)
result = splitter.execute(exp_data)

print("Разбиение выполнено!")
print(f"ID сплиттера: {splitter.id}")

/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


Разбиение выполнено!
ID сплиттера: AASplitter┴rs 21┴


## 3. Просмотр результатов разбиения

Посмотрим на созданную колонку с метками групп и размеры групп.

In [77]:
# Колонка с метками групп сохраняется в additional_fields
split_column = result.additional_fields[splitter.id]
print("Колонка с метками групп:")
print(split_column.value_counts())

Колонка с метками групп:


/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/base.py:1437: FutureWarning: The resulting Series will have a fixed name of 'count' from 4.0.0.
  warnings.warn(
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


  AASplitter┴rs 21┴  count
0           control   5000
1            test_0   5000

2 rows × 2 columns


In [78]:
# # Используем GroupSizes для подсчёта размеров групп
# group_sizes = GroupSizes(grouping_role=AdditionalTreatmentRole())
# sizes_result = group_sizes.execute(result)

# print("\nРазмеры групп:")
# print(sizes_result.analysis_tables[group_sizes.id])

In [109]:
(
    ds.data
    # .toPandas()
    .groupby("age")
    .apply(udf_func)
    # .agg(lambda x: {col : x.value_counts() for col in ds.data.columns})
)

/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If the type hints is not specified for `groupby.apply`, it is expensive to infer the data type internally.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
26/03/25 14:00:45 ERROR Executor: Exception in task 0.0 in stage 202.0 (TID 404)
org.apache.spark.api.python.PythonException: Traceback (most recent call last):
  File "/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/groupby.py", line 2307, in rename_output
    pdf.columns = return_schema.names
  File "/home/eric/.local/lib/python3.8/site-packages/pandas/core/generic.py", line 6004, in __setattr__
    pass
  File "pandas/_libs/properties.pyx", line 69, in pandas._libs.properties.AxisProperty.__set__
  File "/home/eric/.local/lib/python3.8/site-packages/pandas/core/generic.py", line 730, in _set_axis
    self._mgr.set_axis(axis, labels)
  File "/home/eric/.local/lib/python3.8/site-packages/pandas/core/interna

PythonException: 
  An exception was thrown from the Python worker. Please see the stack trace below.
Traceback (most recent call last):
  File "/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/groupby.py", line 2307, in rename_output
    pdf.columns = return_schema.names
  File "/home/eric/.local/lib/python3.8/site-packages/pandas/core/generic.py", line 6004, in __setattr__
    pass
  File "pandas/_libs/properties.pyx", line 69, in pandas._libs.properties.AxisProperty.__set__
  File "/home/eric/.local/lib/python3.8/site-packages/pandas/core/generic.py", line 730, in _set_axis
    self._mgr.set_axis(axis, labels)
  File "/home/eric/.local/lib/python3.8/site-packages/pandas/core/internals/managers.py", line 225, in set_axis
    self._validate_set_axis(axis, new_labels)
  File "/home/eric/.local/lib/python3.8/site-packages/pandas/core/internals/base.py", line 70, in _validate_set_axis
    raise ValueError(
ValueError: Length mismatch: Expected axis has 10 elements, new values

26/03/25 14:00:46 ERROR Executor: Exception in task 0.0 in stage 205.0 (TID 406)
org.apache.spark.api.python.PythonException: Traceback (most recent call last):
  File "/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/groupby.py", line 2307, in rename_output
    pdf.columns = return_schema.names
  File "/home/eric/.local/lib/python3.8/site-packages/pandas/core/generic.py", line 6004, in __setattr__
    pass
  File "pandas/_libs/properties.pyx", line 69, in pandas._libs.properties.AxisProperty.__set__
  File "/home/eric/.local/lib/python3.8/site-packages/pandas/core/generic.py", line 730, in _set_axis
    self._mgr.set_axis(axis, labels)
  File "/home/eric/.local/lib/python3.8/site-packages/pandas/core/internals/managers.py", line 225, in set_axis
    self._validate_set_axis(axis, new_labels)
  File "/home/eric/.local/lib/python3.8/site-packages/pandas/core/internals/base.py", line 70, in _validate_set_axis
    raise ValueError(
ValueError: Length mismatch: Expected axis has

PythonException: 
  An exception was thrown from the Python worker. Please see the stack trace below.
Traceback (most recent call last):
  File "/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/groupby.py", line 2307, in rename_output
    pdf.columns = return_schema.names
  File "/home/eric/.local/lib/python3.8/site-packages/pandas/core/generic.py", line 6004, in __setattr__
    pass
  File "pandas/_libs/properties.pyx", line 69, in pandas._libs.properties.AxisProperty.__set__
  File "/home/eric/.local/lib/python3.8/site-packages/pandas/core/generic.py", line 730, in _set_axis
    self._mgr.set_axis(axis, labels)
  File "/home/eric/.local/lib/python3.8/site-packages/pandas/core/internals/managers.py", line 225, in set_axis
    self._validate_set_axis(axis, new_labels)
  File "/home/eric/.local/lib/python3.8/site-packages/pandas/core/internals/base.py", line 70, in _validate_set_axis
    raise ValueError(
ValueError: Length mismatch: Expected axis has 10 elements, new values

# Тестируем стат.тесты

In [6]:
test = StatsChi2Test(
    grouping_role=AdditionalTreatmentRole(),
    target_roles=TargetRole(),
    reliability=0.05
)
result_test = test.execute(result)

/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


AnalysisException: [UNRESOLVED_ROUTINE] Cannot resolve function `value_counts` on search path [`system`.`builtin`, `system`.`session`, `spark_catalog`.`default`].; line 1 pos 0

In [80]:
test = StatsTTest(
    grouping_role=AdditionalTreatmentRole(),
    target_roles=TargetRole(),
    reliability=0.05
)
result_test = test.execute(result)

/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is l

pandas_backend dict = {'data': {'mean┆pre_spends': 487.30713810684995, 'std┆pre_spends': 18.708522253560385, 'count┆pre_spends': 4511}}
pandas_backend dict = {'data': {'mean┆pre_spends': 487.56325167037863, 'std┆pre_spends': 19.322387347936715, 'count┆pre_spends': 4490}}
pandas_backend dict = {'data': {'p-value': 0.0053452901695765645, 'statistic': 2.7861314523516674, 'pass': True}}


In [9]:
# Просмотр результатов т-теста
print("Результаты т-теста:")
for table_id, table in result_test.analysis_tables.items():
    print(f"\nТаблица {table_id}:")
    print(table)

Результаты т-теста:

Таблица StatsTTest┴┴pre_spends┆stats:
         mean┆pre_spends  std┆pre_spends  count┆pre_spends
control       487.307138       18.708522            4511.0
test_0        487.563252       19.322387            4490.0

2 rows × 3 columns

Таблица StatsTTest┴┴pre_spends:
                    p-value  statistic  pass
test_0┆pre_spends  0.005345   2.786131   1.0

1 rows × 3 columns


## 4. Доступ к группам

Если `save_groups=True`, группы сохраняются в `result.groups` для удобного доступа.

In [ ]:
# Прямой доступ к группам
print("Доступные группы:", list(result.groups.keys()))

# Получаем данные по имени группы
control_group = result.groups[splitter.id]['control']
test_group = result.groups[splitter.id]['test_0']

print(f"\nControl группа: {len(control_group)} записей")
print(f"Test группа: {len(test_group)} записей")

In [ ]:
# Сравниваем средние значения метрик между группами
print("\nСредние значения в control группе:")
print(control_group[['pre_spends', 'post_spends']].mean())

print("\nСредние значения в test группе:")
print(test_group[['pre_spends', 'post_spends']].mean())

## 5. Несколько разбиений с разными random_state

Покажем, как можно запустить несколько итераций с разными seed.

In [ ]:
# Запускаем 5 разбиений с разными random_state
for rs in [10, 20, 30, 40, 50]:
    splitter = AASplitter(control_size=0.5, random_state=rs, save_groups=False)
    result = splitter.execute(ExperimentData(data))
    
    split_col = result.additional_fields[splitter.id]
    counts = split_col.value_counts()
    
    print(f"random_state={rs}: control={counts.get('control', 0)}, test_0={counts.get('test_0', 0)}")

## 6. Разбиение на несколько групп

Можно указать `groups_sizes` для разбиения на более чем 2 группы.

In [ ]:
# Разбиение на 3 группы: 50% control, 30% test_0, 20% test_1
splitter = AASplitter(
    groups_sizes=[0.5, 0.3, 0.2],
    random_state=42,
    save_groups=True
)
result = splitter.execute(ExperimentData(data))

split_col = result.additional_fields[splitter.id]
print("Разбиение на 3 группы:")
print(split_col.value_counts())

print("\nДоступные группы:", list(result.groups[splitter.id].keys()))

## Итоги

`AASplitter` выполняет:
1. Создаёт колонку с метками групп в `additional_fields`
2. Опционально сохраняет группы в `data.groups` для удобного доступа
3. Поддерживает кастомные размеры групп через `groups_sizes`
4. Используется как первый шаг в AA/AB тестах